# Using Gemini for Video Analysis

Analyse a video with Google's native Python SDK, ask questions with timestamped evidence, and extract validated structured outputs.

This is the practical companion to **Using Gemini for Video Analysis**. Follow the deck's sequence: **upload once, wait for ACTIVE, then ask**. We use `google-genai` and the Interactions API throughout.

By the end, you can reuse one uploaded video, continue a conversation, turn observations into a Pydantic model, and choose a sampling strategy for your task.

## 1. Why Gemini is competitive for video

Gemini accepts a video file and processes its visual and audio content together. That makes questions such as "What happens while the presenter explains this step?" straightforward to implement.

| Provider and API | Route to video analysis | Work your application handles |
| --- | --- | --- |
| Gemini Interactions | Send the video directly | Upload and reference the file, choose processing options |
| OpenAI Responses with GPT-6.1 Sol | Send images of selected frames and any transcript | Extract frames, preserve timestamps, obtain and align a transcript |
| Claude Messages vision | Send images of selected frames and any transcript | Extract frames, preserve timestamps, obtain and align a transcript |

GPT-6.1 Sol's model page lists image input, with audio and video unsupported. Claude's vision guide describes image inputs. Both can reason about supplied frames, but their APIs in this comparison need preprocessing. This comparison concerns these developer APIs; consumer apps and agent tools can provide different workflows.

Sources: [Gemini video](https://ai.google.dev/gemini-api/docs/video-understanding), [GPT-6.1 Sol modalities](https://developers.openai.com/api/docs/models/gpt-6.1-sol), [OpenAI vision](https://developers.openai.com/api/docs/guides/images-vision), [Claude vision](https://platform.claude.com/docs/en/build-with-claude/vision).

### Price is one part of the decision

Standard paid API rates in USD per million tokens, checked **8 October 2026**:

| Model | Uncached input | Output |
| --- | ---: | ---: |
| Gemini 3.8 Flash | $0.75 | $3.75 |
| GPT-6.1 Sol | $2.00 | $10.00 |
| Claude Sonnet 5.5 | $2.00 | $10.00 |
| Claude Haiku 5.5, prompts up to 100,000 tokens | $0.10 | $0.50 |

Gemini's listed promotional rates end on 31 December 2026, then rise to USD 1.50 input and USD 7.50 output. Haiku's rates rise above the stated prompt threshold. These prices are **not an equal-quality benchmark or a cost-per-video comparison**: providers tokenise media differently, reasoning consumes tokens, and caching changes cost. OpenAI also changes rates for long prompts.

Our practical conclusion: Gemini is a strong starting point when native video input saves preprocessing and its answers pass your checks. For a handful of still images or an existing transcript, another model may cost less. Measure correctness, latency and the cost of the complete workflow on your own clips.

Sources: [Google pricing](https://ai.google.dev/gemini-api/docs/pricing), [OpenAI model pricing](https://developers.openai.com/api/docs/models/gpt-6.1-sol), [Claude pricing](https://platform.claude.com/docs/en/about-claude/pricing).

## 2. Why Google's native SDK

The [Files API](https://ai.google.dev/gemini-api/docs/files) stores uploads for **48 hours**, with a **2 GB per-file** limit. Reuse the URI across questions instead of sending the file bytes each time. Google recommends the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions-overview) for new projects, and it supports conversations through `previous_interaction_id`.

There are three separate kinds of reuse:

| Mechanism | What it saves or preserves |
| --- | --- |
| Uploaded file URI | Repeated transfer of the video bytes |
| `previous_interaction_id` | Conversation history on the server |
| Prompt caching | Inference work and token charges when a cache actually applies |

**One upload does not mean one input-token charge.** Each analysis can process video or conversation context again. Look at the returned usage.

A gateway can also support video. For example, [OpenRouter documents video URLs and base64 input](https://openrouter.ai/docs/guides/overview/multimodal/videos). Use a gateway for a shared provider interface when it supports your needs. Here the native SDK exposes file management and Gemini's conversation and processing controls directly.

## 3. Setup

Run in **Jupyter or Google Colab with Python 3.10+**. Create a [Gemini API key in Google AI Studio](https://aistudio.google.com/apikey). We use a hidden `getpass` prompt if `GEMINI_API_KEY` is missing.

The walkthrough makes **three generation requests**: a summary, a follow-up, and a structured timeline. Dense sampling adds one request when enabled. The five-minute practice adds one structured-output request. Paid accounts incur charges. No OpenAI or Anthropic key is needed.

The SDK version below keeps the demonstrated API reproducible. If you change it, check the linked documentation and restart your kernel if packages were already imported.

In [ ]:
"""Analyse video with Gemini and validate timestamped, structured answers."""

# Pylint treats notebook cells as one script and replaces pip magics with pass.
# pylint: disable=wrong-import-position,unnecessary-pass
%pip install "google-genai==2.29.0" "pydantic>=2.9,<3" pandas --upgrade --quiet

In [ ]:
MODEL = "gemini-3.8-flash"

# Leave empty to download Google's public pottery sample, about 12 seconds / 70 MB.
# Or set a path to an MP4 in your Jupyter or Colab runtime.
VIDEO_PATH = ""
SAMPLE_URL = "https://storage.googleapis.com/generativeai-downloads/videos/Pottery.mp4"

RUN_DENSE_SAMPLING = False

In [ ]:
import json
import os
import shutil
import tempfile
import time
from getpass import getpass
from pathlib import Path
from typing import Literal
from urllib.request import urlopen

import pandas as pd
from google import genai
from google.genai.interactions import Interaction
from IPython.display import Markdown, Video, display
from pydantic import BaseModel, Field, model_validator

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

INSTRUCTIONS = (
    "Analyse only the supplied video. Separate visible evidence from audible evidence. "
    "Use MM:SS timestamps for prose answers. State when something cannot be observed. "
    "Avoid guessing identities, intentions or events outside the video."
)
GENERATION_CONFIG = {"max_output_tokens": 4096, "thinking_level": "low"}
requests_made: list[tuple[str, Interaction]] = []

## 4. Choose and watch the video

The default downloads a short public pottery clip hosted by Google. Watch it before asking the model questions so you can check its answers. Replace `VIDEO_PATH` in the configuration cell to analyse your own recording. In Colab, use the Files sidebar to upload an MP4, then copy its runtime path into that cell.

The sample preview streams the public URL. For your own file the preview embeds its bytes in the cell output, so use a short clip and clear that output before sharing the notebook.

In [5]:
if VIDEO_PATH:
    video_path = Path(VIDEO_PATH).expanduser()
else:
    # Store the sample in the runtime's temporary directory, outside the repository.
    video_path = Path(tempfile.gettempdir()) / "gemini_video_analysis_pottery.mp4"
    if not video_path.exists():
        print("Downloading the public sample...")
        temporary_path = video_path.with_suffix(".download")
        try:
            with urlopen(SAMPLE_URL, timeout=60) as source, temporary_path.open("wb") as target:
                shutil.copyfileobj(source, target)
            temporary_path.replace(video_path)
        finally:
            temporary_path.unlink(missing_ok=True)

if not video_path.is_file() or video_path.stat().st_size == 0:
    raise FileNotFoundError(f"Choose an existing, non-empty video: {video_path}")
if video_path.suffix.lower() != ".mp4":
    raise ValueError("This introductory example expects an MP4 file.")
if video_path.stat().st_size > 2 * 1024**3:
    raise ValueError("The Files API has a 2 GB per-file limit. Use a shorter MP4.")

print(f"{video_path.name}: {video_path.stat().st_size / 1_000_000:.1f} MB")
if VIDEO_PATH:
    display(Video(filename=str(video_path), embed=True, width=640))
else:
    display(Video(url=SAMPLE_URL, width=640))

gemini_video_analysis_pottery.mp4: 70.3 MB


## 5. Upload once, then wait for ACTIVE

An upload can return while Google's video processing is still running. The loop below waits for `ACTIVE`, reports failed processing, and has a five-minute deadline. After a timeout you can retrieve the existing file by its name and check its state again.

In [6]:
video_file = client.files.upload(
    file=str(video_path),
    config={"mime_type": "video/mp4", "display_name": video_path.name},
)
video_file_name = video_file.name
if not video_file_name:
    raise RuntimeError("The upload did not return a file name.")
print("Uploaded:", video_file_name)

deadline = time.monotonic() + 300
while video_file.state and video_file.state.name == "PROCESSING":
    if time.monotonic() >= deadline:
        raise TimeoutError(
            f"Processing still running. Check client.files.get(name={video_file_name!r})."
        )
    time.sleep(2)
    video_file = client.files.get(name=video_file_name)

if not video_file.state or video_file.state.name != "ACTIVE":
    raise RuntimeError(f"Video is not ready: {video_file.state}. Details: {video_file.error}")

print("Ready:", video_file.state.name)
video_input = {
    "type": "video",
    "uri": video_file.uri,
    "mime_type": video_file.mime_type,
    "processing": "static",
    "resolution": "low",
}

Uploaded: files/idqnzpksod1g
Ready: ACTIVE


## 6. Ask a question with timestamps

Static processing samples **one frame per second by default**, alongside audio. At low resolution Google estimates roughly **100 tokens per second**, so a ten-minute video can use around 60,000 input tokens before the prompt and answer. Fast actions can fall between sampled frames.

Place the video before the question. The response's `output_text` contains the answer. We explicitly keep this interaction so the next cell can continue the conversation.

Source: [Video processing and tokens](https://ai.google.dev/gemini-api/docs/video-understanding).

In [7]:
summary = client.interactions.create(
    model=MODEL,
    system_instruction=INSTRUCTIONS,
    generation_config=GENERATION_CONFIG,
    input=[
        video_input,
        {"type": "text", "text": (
            "Summarise the clip in three short bullets. Cite a MM:SS timestamp for each. "
            "Describe the visible actions and mention any audio you can actually hear."
        )},
    ],
    store=True,
    stream=False,
)
if not isinstance(summary, Interaction):
    raise TypeError("Expected a non-streaming Interaction response.")
requests_made.append(("Summary", summary))
summary_id = summary.id
if not summary_id:
    raise RuntimeError("The stored summary did not return an interaction ID.")
print("Stored interaction:", summary_id)

if summary.status != "completed" or not summary.output_text:
    raise RuntimeError(
        f"No completed answer: status={summary.status}, errors={summary.errors}"
    )
display(Markdown(summary.output_text))

Stored interaction: v1_ChdTc19JYXAyaEFlS3Z4TjhQOW9Id2dBNBIXU3NfSWFwMmhBZUt2eE44UDlvSHdnQTQ


Here is a summary of the clip based on visible and audible evidence:

* **00:00 - 00:03:** The video opens showing an arrangement of handmade ceramic tumblers and bowls on a surface, each grouping accompanied by handwritten labels displaying names, dimensions, and prices. In the background, music plays along with ambient voices talking.
* **00:03 - 00:07:** The camera pans and zooms in closer toward the ceramic tumblers labeled "#5 Artichoke double dip" and then shifts toward the adjacent "small bowls" marked at $35, accompanied by continuing music and chatter.
* **00:07 - 00:11:** The camera moves right to focus on two "med bowls" priced at $40, then pans down to frame a glaze test tile and its label reading "#6 gemini double dip SLOW COOL" while the background audio continues.

### Follow-up Q&A with conversation history

`previous_interaction_id` points to the previous turn. You can ask a follow-up without supplying the video again. Repeat your instructions and generation settings: those apply to each interaction individually.

The history can still contribute to token usage. For an independent question, send `[video_input, question]` again using the same uploaded URI.

Source: [Interactions conversation state](https://ai.google.dev/gemini-api/docs/interactions-overview).

In [8]:
follow_up = client.interactions.create(
    model=MODEL,
    system_instruction=INSTRUCTIONS,
    generation_config=GENERATION_CONFIG,
    previous_interaction_id=summary_id,
    input="What changes between the beginning and the end? Cite moments that support your answer.",
    store=False,
    stream=False,
)
if not isinstance(follow_up, Interaction):
    raise TypeError("Expected a non-streaming Interaction response.")
requests_made.append(("Follow-up Q&A", follow_up))

if follow_up.status != "completed" or not follow_up.output_text:
    raise RuntimeError(
        f"No completed answer: status={follow_up.status}, errors={follow_up.errors}"
    )
display(Markdown(follow_up.output_text))

Between the beginning and the end of the video, the visible framing shifts entirely:

* **At the beginning (00:00 - 00:01):** The camera shows a wide shot of the entire display table, capturing a stack of ceramic tumblers on the left, several small bowls in the center, and larger bowls on the right, along with all associated price tags.
* **At the end (00:09 - 00:11):** The camera has moved down and to the right into a tight close-up, cutting out the tumblers and small bowls completely to focus solely on a ceramic glaze test tile labeled "6 G 6" and a paper slip inscribed "#6 gemini double dip SLOW COOL."

## 7. Structured outputs: video into Python data

A prose summary helps a reader. A typed timeline lets your application build chapters, search observations or prepare a review table.

Define the expected shape with Pydantic, pass `model_json_schema()` as the API's `response_format`, then validate the returned JSON with `model_validate_json()`. The `model_validator` checks that an event ends after it starts. The API schema alone cannot enforce that relationship.

We use numeric seconds for data and convert them to MM:SS for display. `audio_evidence` and `uncertainty` accept `None`. A schema can enforce shape and types; **it cannot establish that an observation is true**. Watch the cited moments to verify them.

Source: [Gemini structured outputs](https://ai.google.dev/gemini-api/docs/structured-output). Gemini supports a subset of JSON Schema, so keep schemas small.

In [9]:
class VideoEvent(BaseModel):
    """A timestamped interval with visible and audible evidence."""

    start_seconds: float = Field(
        ge=0, description="Event start, in seconds from the original video's start."
    )
    end_seconds: float = Field(
        ge=0, description="Event end, in seconds from the original video's start."
    )
    visual_evidence: str = Field(description="What is directly visible during this interval.")
    audio_evidence: str | None = Field(
        description="What is directly audible, or null if there is no relevant audio."
    )
    uncertainty: str | None = Field(description="Any uncertainty in the observation, or null.")

    @model_validator(mode="after")
    def check_interval(self) -> "VideoEvent":
        """Reject intervals that do not end after they start."""
        if self.end_seconds <= self.start_seconds:
            raise ValueError("Each event must end after it starts.")
        return self


class VideoAnalysis(BaseModel):
    """A short summary and a bounded timeline of video events."""

    title: str
    summary: str
    events: list[VideoEvent] = Field(min_length=1, max_length=5)


structured = client.interactions.create(
    model=MODEL,
    system_instruction=INSTRUCTIONS,
    generation_config=GENERATION_CONFIG,
    input=[video_input, {"type": "text", "text": (
        "Extract a short title, a one-sentence summary and 1 to 5 key events. "
        "Use seconds relative to the original video. Every interval must fit within the video. "
        "Separate visual from audio evidence, and record uncertainty."
    )}],
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": VideoAnalysis.model_json_schema(),
    },
    store=False,
    stream=False,
)
if not isinstance(structured, Interaction):
    raise TypeError("Expected a non-streaming Interaction response.")
requests_made.append(("Structured timeline", structured))

if structured.status != "completed" or not structured.output_text:
    raise RuntimeError(
        f"No completed JSON: status={structured.status}, errors={structured.errors}"
    )
analysis = VideoAnalysis.model_validate_json(structured.output_text)
print(analysis.model_dump_json(indent=2))

{
  "title": "Display of Handcrafted Pottery and Labels",
  "summary": "A camera pans across a tabletop showcasing ceramic tumblers and bowls alongside handwritten price and dimension labels.",
  "events": [
    {
      "start_seconds": 0.0,
      "end_seconds": 5.0,
      "visual_evidence": "The camera starts showing an arrangement of ceramic tumblers and brown bowls on a surface, then pans and tilts closer towards stacked tumblers and a torn paper slip reading '#5 Artichoke double dip TUMBLERS $20 4\"h x 3\"d -ish'.",
      "audio_evidence": null,
      "uncertainty": null
    },
    {
      "start_seconds": 5.0,
      "end_seconds": 11.0,
      "visual_evidence": "The camera pans to the right, showing two small brown ceramic bowls next to a label marked 'small bowls 3.5\"h x 6.5\"d $35', followed by medium brown ceramic bowls with a label reading 'med bowls 4\"h x 7\"d $40', ending near a test tile and slip labeled '#6 gemini double dip SLOW COOL'.",
      "audio_evidence": null,
  

In [10]:
def timestamp(seconds: float) -> str:
    """Convert seconds to an MM:SS timestamp for display."""
    minutes, remaining = divmod(int(seconds), 60)
    return f"{minutes:02d}:{remaining:02d}"


timeline = pd.DataFrame([
    {
        "Start": timestamp(event.start_seconds),
        "End": timestamp(event.end_seconds),
        "Visible evidence": event.visual_evidence,
        "Audio evidence": event.audio_evidence,
        "Uncertainty": event.uncertainty,
    }
    for event in analysis.events
])
display(timeline)

,Start,End,Visible evidence,Audio evidence,Uncertainty
0,00:00,00:05,The camera starts showing an arrangement of ce...,None,None
1,00:05,00:11,"The camera pans to the right, showing two smal...",None,None


## 8. Optional: inspect a short interval more closely

Enable `RUN_DENSE_SAMPLING` and rerun this cell to inspect the first five seconds at three frames per second. This adds one request. More samples may capture a brief movement, and also consume more tokens. Use static processing for explicit clipping and FPS controls. In this SDK, offsets use duration strings such as `"5s"`.

For another clip, choose a relevant interval inside its duration. Compare against section 6 and replay the interval yourself. Higher FPS does not guarantee accurate timing.

In [12]:
RUN_DENSE_SAMPLING = True

In [13]:
if RUN_DENSE_SAMPLING:
    detailed_input = {
        **video_input,
        "processing": {"type": "static", "start_offset": "0s", "end_offset": "5s", "fps": 3},
    }
    detailed = client.interactions.create(
        model=MODEL,
        system_instruction=INSTRUCTIONS,
        generation_config=GENERATION_CONFIG,
        input=[detailed_input, {"type": "text", "text": (
            "Describe the visible actions in this interval in their order. "
            "Use timestamps relative to the original video and state any uncertainty."
        )}],
        store=False,
        stream=False,
    )
    if not isinstance(detailed, Interaction):
        raise TypeError("Expected a non-streaming Interaction response.")
    requests_made.append(("Dense five-second interval", detailed))
    if detailed.status != "completed" or not detailed.output_text:
        raise RuntimeError(
            f"No completed answer: status={detailed.status}, errors={detailed.errors}"
        )
    display(Markdown(detailed.output_text))
else:
    print("Dense sampling is disabled. Set RUN_DENSE_SAMPLING = True to try it.")

Here is a chronological description of the visible actions during this interval:

- **00:00 - 00:02**: The camera views a display of ceramic items arranged on a surface, showing glazed tumblers on the left, bowls in the center, and larger bowls to the right, each accompanied by small handwritten specification/price cards.
- **00:02 - 00:05**: The camera slowly pans and moves forward/downward toward the stack and grouping of tumblers and their handwritten card labeled "#5 Artichoke double dip TUMBLERS $20". No physical movement of the pottery or objects occurs.

## 9. Inspect the actual token usage

The Interactions API returns `usage`. Compare the summary and follow-up to see that a shorter request payload does not mean zero input tokens. Cached tokens are a subset of input tokens, so avoid adding them to the input count again. Thought and tool-use fields help explain the work behind an answer.

This table shows each response's reported counts. Use your provider's billing view for the final charge. For model selection, record full-workflow cost, latency, correctly answered questions, false claims and timestamp accuracy on the same test clips.

In [14]:
usage_rows = []
for label, interaction in requests_made:
    usage = interaction.usage
    usage_rows.append({
        "Request": label,
        "Input tokens": usage.total_input_tokens if usage else None,
        "Cached tokens": usage.total_cached_tokens if usage else None,
        "Output tokens": usage.total_output_tokens if usage else None,
        "Thought tokens": usage.total_thought_tokens if usage else None,
        "Tool-use tokens": usage.total_tool_use_tokens if usage else None,
        "Total tokens": usage.total_tokens if usage else None,
    })
display(pd.DataFrame(usage_rows))

,Request,Input tokens,Cached tokens,Output tokens,Thought tokens,Tool-use tokens,Total tokens
0,Summary,1158,0,196,0,0,1354
1,Follow-up Q&A,1143,0,158,0,0,1301
2,Structured timeline,1172,0,293,0,0,1465
3,Dense five-second interval,1182,0,135,0,0,1317


## 10. Clean up your uploaded video

Run this after all questions. It deletes only this lesson's uploaded file. To analyse it again after closing the client, rerun the setup and upload cells. The Files API otherwise expires it automatically after 48 hours.

File storage and interaction history are separate. The first summary uses a stored interaction. We delete it too after finishing with the conversation.

In [15]:
if not video_file_name:
    raise RuntimeError("No uploaded file name is available for cleanup.")
client.files.delete(name=video_file_name)
print("Deleted the lesson upload:", video_file_name)

# Delete the interaction we explicitly stored for conversation support.
if not summary_id:
    raise RuntimeError("No stored interaction ID is available for cleanup.")
client.interactions.delete(summary_id)
print("Deleted the stored lesson interaction.")
client.close()

Deleted the lesson upload: files/idqnzpksod1g
Deleted the stored lesson interaction.


## 11. Your turn: a five-minute video challenge

We've walked through the code together. Now it's your turn to upload a video, extract structured data, and check the evidence for yourself.

**Your goal:** produce one validated observation of a visible action, with a timestamp and evidence you can verify. Use the sample already saved at `video_path` to keep this exercise short. You can use another short MP4 if it is already in your runtime.

| Time | Your task | What you should have |
| --- | --- | --- |
| 0:00-1:00 | Watch a moment and write down what you see | One reference action and its approximate timestamp |
| 1:00-2:00 | Define a small Pydantic model | A clear contract for the answer |
| 2:00-4:00 | Upload the clip and request structured output | A validated Python object |
| 4:00-5:00 | Replay the timestamp and compare the evidence | A checked observation, or a clear reason to reject it |

Pause the timer while the upload is processing. The core exercise makes **one generation request**; a retry adds another.

### Utilities

Run the next cell once before starting the timer. These functions use the same native SDK calls as the walkthrough and handle processing, response checks and JSON validation for you.

| Function | What it does |
| --- | --- |
| `upload_ready_video(client, file_path)` | Uploads a local MP4 and waits for `ACTIVE` |
| `analyse_video_structured(client, uploaded, prompt=..., schema=..., model=...)` | Requests your schema and returns `(validated_result, response)` |
| `delete_uploaded_video(client, uploaded)` | Deletes the file you uploaded for this exercise |

Section 10 deleted the earlier upload and closed its client. Create a fresh client for this exercise; keep using the same `GEMINI_API_KEY`.

In [ ]:
from typing import TypeVar

from google.genai import types

PracticeResult = TypeVar("PracticeResult", bound=BaseModel)


def upload_ready_video(
    api_client: genai.Client,
    file_path: str | Path,
    *,
    timeout_seconds: float = 300,
) -> types.File:
    """Upload a local MP4 and return its file only after processing is ACTIVE."""
    local_path = Path(file_path).expanduser()
    if not local_path.is_file() or local_path.stat().st_size == 0:
        raise FileNotFoundError(f"Choose an existing, non-empty video: {local_path}")
    if local_path.suffix.lower() != ".mp4":
        raise ValueError("Choose an MP4 for this exercise.")
    if local_path.stat().st_size > 2 * 1024**3:
        raise ValueError("Use a video smaller than the Files API's 2 GB limit.")
    if timeout_seconds <= 0:
        raise ValueError("timeout_seconds must be greater than zero.")

    uploaded = api_client.files.upload(
        file=str(local_path),
        config={"mime_type": "video/mp4", "display_name": local_path.name},
    )
    file_name = uploaded.name
    if not file_name:
        raise RuntimeError("The upload did not return a file name.")

    deadline = time.monotonic() + timeout_seconds
    while uploaded.state and uploaded.state.name == "PROCESSING":
        if time.monotonic() >= deadline:
            raise TimeoutError(f"Video is still processing: {file_name}. Try a shorter clip.")
        time.sleep(2)
        uploaded = api_client.files.get(name=file_name)

    if not uploaded.state or uploaded.state.name != "ACTIVE":
        raise RuntimeError(f"Video processing failed: {uploaded.state}. {uploaded.error}")
    if not uploaded.uri:
        raise RuntimeError("The ready video did not return a URI.")
    # Poll responses can omit the name; preserve it for the later delete call.
    uploaded.name = file_name
    return uploaded


def analyse_video_structured(
    api_client: genai.Client,
    uploaded: types.File,
    *,
    prompt: str,
    schema: type[PracticeResult],
    model: str,
) -> tuple[PracticeResult, Interaction]:
    """Make one video request and return validated data plus the usage response."""
    if not uploaded.state or uploaded.state.name != "ACTIVE" or not uploaded.uri:
        raise ValueError("Upload the video and wait for ACTIVE before asking a question.")
    if not prompt.strip():
        raise ValueError("Write a prompt describing what you want to extract.")

    response = api_client.interactions.create(
        model=model,
        system_instruction=(
            "Use only evidence from the supplied video. Use seconds from its start. "
            "Describe visible observations, and record uncertainty instead of guessing."
        ),
        generation_config={"max_output_tokens": 1024, "thinking_level": "low"},
        input=[
            {
                "type": "video",
                "uri": uploaded.uri,
                "mime_type": uploaded.mime_type or "video/mp4",
                "processing": "static",
                "resolution": "low",
            },
            {"type": "text", "text": prompt},
        ],
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": schema.model_json_schema(),
        },
        store=False,
        stream=False,
    )
    if not isinstance(response, Interaction):
        raise TypeError("Expected a non-streaming Interaction response.")
    if response.status != "completed" or not response.output_text:
        raise RuntimeError(f"No completed JSON: {response.status}. {response.errors}")
    return schema.model_validate_json(response.output_text), response


def delete_uploaded_video(api_client: genai.Client, uploaded: types.File) -> None:
    """Delete this exercise's upload using its validated file name."""
    file_name = uploaded.name
    if not file_name:
        raise ValueError("The uploaded video has no file name to delete.")
    api_client.files.delete(name=file_name)

### 1. Observe first, then define the contract

Before asking Gemini, write down one visible action and approximately when it happens. Keep that reference beside you so you can judge the answer independently.

In the blank cell, define `Observation(BaseModel)` with these four fields:

| Field | Type | What it must contain |
| --- | --- | --- |
| `action` | `str` | One short description of a visible action |
| `timestamp_seconds` | `float`, using `Field(ge=0)` | A moment where the action is visible |
| `visible_evidence` | `str` | A concrete detail visible at that moment |
| `uncertainty` | `str | None` | What cannot be established, or `None` |

Keep the schema small. You are extracting **one observation**, not recreating the whole timeline.

### 2. Upload, request and inspect

Use the next blank cell to put the steps together:

1. Create a fresh client with `with genai.Client(api_key=os.environ["GEMINI_API_KEY"]) as practice_client:`. The context manager closes it when you finish.
2. Call `upload_ready_video(practice_client, video_path)` and save the returned file as `practice_file`.
3. Write your own prompt asking for **one visible action**, timestamped evidence, and uncertainty. Focus it on the same short interval you watched, without telling Gemini your reference answer.
4. Inside a `try` block, call `analyse_video_structured(...)` with your prompt, `schema=Observation` and `model=MODEL`. Save both returned values as `result, response`.
5. Print `result.model_dump_json(indent=2)` and `response.usage`. In a `finally` block, call `delete_uploaded_video(practice_client, practice_file)` so the upload is deleted even if the request or validation fails.

After uploading, keep the request and cleanup inside the client context. You can reuse `practice_file` for another question before cleanup; each new request can incur input-token charges.

### 3. Check the answer against the clip

You are finished when you can answer **yes** to each question:

- Did the response validate as your `Observation` model?
- Is its timestamp within the video, and can you see the claimed action at that moment?
- Does `visible_evidence` describe a specific thing you can point to, rather than a guess about intent?
- Does the answer admit uncertainty where the clip cannot establish something?

Compare it with the reference you wrote before the request. **A valid schema does not make an observation true.** If the evidence is wrong, record why and change one instruction in your prompt before an optional retry.

**Explain your result in one sentence:** "I accepted/rejected this observation because at ___ seconds I could/could not see ___."

**Optional extension after the five minutes:** add `tool_or_object: str | None` to your model, update the prompt, and rerun with a fresh client and upload. Confirm the new field appears in the validated result.

## Summary

- Upload once, wait for `ACTIVE`, then reference the file URI in your questions.
- Use a Pydantic schema to request structured output, then validate the returned JSON.
- Check timestamped evidence against the actual clip: correct types and credible observations are separate checks.
- Inspect usage, delete uploads when finished, and close the client.